In [1]:
import sqlite3
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import ollama
from __future__ import annotations

import os
import random
from dataclasses import dataclass
from pathlib import Path
from typing import Any

from steam_sqlite import load_games_from_sqlite


In [2]:
# Load the model 
model = SentenceTransformer('all-mpnet-base-v2')
DB_PATH = 'raglooker/steam_games_reviews_25.sqlite'

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [3]:
# current working directory
print("Current Directory:", os.getcwd())

# check for file
file_name = 'steam_games_reviews_25.sqlite'
if os.path.exists(file_name):
    print(f"{file_name} found.")
else:
    print(f"Error: {file_name} NOT found in this folder.")
    print("Files actually in this folder:", os.listdir('.'))

Current Directory: C:\Users\athul\raglooker
steam_games_reviews_25.sqlite found.


In [4]:
conn = sqlite3.connect('steam_games_reviews_25.sqlite')

try:
    df_sample = pd.read_sql("SELECT * FROM games LIMIT 1", conn)
    print("Correct Column Names:", df_sample.columns.tolist())
except Exception as e:
    print("Error reading table:", e)
finally:
    conn.close()

Correct Column Names: ['appid', 'name', 'release_date', 'required_age', 'price', 'dlc_count', 'detailed_description', 'about_the_game', 'short_description', 'reviews_summary', 'header_image', 'website', 'support_url', 'support_email', 'windows', 'mac', 'linux', 'metacritic_score', 'metacritic_url', 'achievements', 'recommendations', 'notes', 'supported_languages_json', 'full_audio_languages_json', 'packages_json', 'developers_json', 'publishers_json', 'categories_json', 'genres_json', 'screenshots_json', 'movies_json', 'user_score', 'score_rank', 'positive', 'negative', 'estimated_owners', 'average_playtime_forever', 'average_playtime_2weeks', 'median_playtime_forever', 'median_playtime_2weeks', 'discount', 'peak_ccu', 'tags_json', 'raw_json']


Simplest Embedding tested

In [5]:
def prep_data():
    conn = sqlite3.connect('steam_games_reviews_25.sqlite')
    
    query = """
    SELECT 
        appid, 
        name, 
        short_description, 
        genres_json, 
        tags_json,
        reviews_summary 
    FROM games
    """ 
    
    df = pd.read_sql(query, conn)
    conn.close()

    df['combined_text'] = (
        df['name'].fillna('') + " " + 
        df['genres_json'].fillna('') + " " + 
        df['tags_json'].fillna('') + " " + 
        df['short_description'].fillna('') + " " +
        df['reviews_summary'].fillna('')
    )
    
    return df['appid'].values, df['combined_text'].tolist()

app_ids, texts = prep_data()

In [6]:
embeddings = model.encode(texts, show_progress_bar=True, batch_size=32)
np.save('game_embeddings_v1.npy', embeddings)
np.save('game_app_ids_v1.npy', app_ids)

Batches:   0%|          | 0/1225 [00:00<?, ?it/s]

Second Embedding 

In [2]:
def gen_embeddings():
    db_path = 'steam_games_reviews_25.sqlite'
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    query = """
    SELECT 
        g.appid, 
        g.name, 
        COALESCE(g.short_description, ''), 
        COALESCE(GROUP_CONCAT(r.review, ' '), '') as player_feedback
    FROM games g
    LEFT JOIN (
        SELECT appid, review FROM reviews LIMIT 50000
    ) r ON g.appid = r.appid
    GROUP BY g.appid
    """
    
    cursor.execute(query)
    rows = cursor.fetchall()
    
    combined_texts = []
    for row in rows:
        appid, name, desc, feedback = row
        hybrid_text = f"GAME: {name}. DESCRIPTION: {desc}. PLAYER VIBE: {feedback[:1000]}"
        combined_texts.append(hybrid_text)

    embeddings = model.encode(combined_texts, show_progress_bar=True, batch_size=32)

    np.save('game_embeddings_v2.npy', embeddings)
    conn.close()

if __name__ == "__main__":
    gen_embeddings()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1225 [00:00<?, ?it/s]

Third Embedding

In [3]:
def gen_embeddingsv2():
    db_path = 'steam_games_reviews_25.sqlite'
    model = SentenceTransformer('all-MiniLM-L6-v2')
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    query = """
SELECT 
    g.appid, g.name, g.short_description,
    GROUP_CONCAT(r.review, ' | ') as reviews,
    SUM(CASE WHEN r.voted_up = 1 THEN 1 ELSE 0 END) as pos,
    COUNT(r.recommendationid) as total
FROM games g
LEFT JOIN reviews r ON g.appid = r.appid
GROUP BY g.appid
HAVING total > 5 
"""
    
    cursor.execute(query)
    rows = cursor.fetchall()
    
    combined_texts = []
    for row in rows:
        appid, name, desc, feedback, pos, total = row
        
        # based on review scores
        approval_rate = (pos / total) * 100 if total > 0 else 0
        if total > 1000 and approval_rate > 85: vibe_label = "Classic"
        elif total < 50 and approval_rate > 90: vibe_label = "Niche"
        elif approval_rate < 60: vibe_label = "Mixed"
        else:vibe_label = "Generally Recommended" 
        
        # approval rating 
        hybrid_text = f"NAME: {name} {name}. " \
                      f"DESCRIPTION: {desc} {desc}. " \
                      f"COMMUNITY VIBE: {vibe_label} with {approval_rate:.0f}% approval. " \
                      f"PLAYER REVIEWS: {feedback[:800]}"
                      
        combined_texts.append(hybrid_text)

    embeddings = model.encode(combined_texts, show_progress_bar=True, batch_size=32)

    np.save('game_embeddings_v3.npy', embeddings)
    conn.close()

if __name__ == "__main__":
    gen_embeddingsv2()

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1224 [00:00<?, ?it/s]

Updated Embeddings

In [7]:
def clean_json_list(json_str):
    try:
        data = json.loads(json_str)
        if isinstance(data, list): return " ".join(data)
        if isinstance(data, dict): return " ".join(data.values())
        return str(data)
    except:
        return str(json_str).replace("[", "").replace("]", "").replace('"', "").replace(",", " ")

def gen_v4():
    db_path = 'steam_games_reviews_25.sqlite'
    model = SentenceTransformer('all-mpnet-base-v2')
    
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    query = """
    SELECT 
        g.appid, g.name, g.short_description,
        COALESCE(g.genres_json, '') as genres,
        COALESCE(g.categories_json, '') as categories,
        COALESCE(g.tags_json, '') as tags,
        GROUP_CONCAT(r.review, ' | ') as reviews,
        SUM(CASE WHEN r.voted_up = 1 THEN 1 ELSE 0 END) as pos,
        COUNT(r.recommendationid) as total
    FROM games g
    LEFT JOIN reviews r ON g.appid = r.appid
    GROUP BY g.appid
    HAVING total > 2
    """
    
    cursor.execute(query)
    rows = cursor.fetchall()
    
    combined_texts = []
    appids = []

    for row in rows:
        appid, name, desc, genres_raw, cats_raw, tags_raw, feedback, pos, total = row
        
        genre_tags = clean_json_list(genres_raw)
        cat_tags = clean_json_list(cats_raw)
        user_tags = clean_json_list(tags_raw)
        
        hybrid_text = (
            f"IDENTITY: {name} {name} {name}. "
            f"MECHANICS: {genre_tags} {user_tags} {genre_tags} {user_tags}. "
            f"FEATURES: {cat_tags}. "
            f"PITCH: {desc}. "
            f"COMMUNITY VIBE: {feedback[:400] if feedback else ''}"
        )
        
        combined_texts.append(hybrid_text)
        appids.append(appid)

    embeddings = model.encode(combined_texts, show_progress_bar=True, batch_size=16)

    np.save('game_embeddings_v4.npy', embeddings)
    np.save('game_appids_v4.npy', np.array(appids))
    conn.close()

if __name__ == "__main__":
    gen_v4()

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Encoding 39164 vectors (V4)...


Batches:   0%|          | 0/2448 [00:00<?, ?it/s]